# 01 - Game walkthrough

A tour of the `blackjack` package: cards, scoring, rules, the dealer, strategies, and a full round.
This notebook only imports from `blackjack`; it defines no game logic.

Setup: `pip install -e ".[notebooks]"` from the project root.

## Cards and decks

Pass a seeded `random.Random` and the shuffle is reproducible.

In [ ]:
import random

from blackjack.game.cards import card_value, create_deck

rng = random.Random(42)
deck = create_deck(rng)
print(len(deck), "cards; top three:", deck[-3:])
print("Value of an Ace:", card_value({"rank": "Ace", "suit": "Hearts"}))

## Scoring hands

Aces count 11 unless that would bust the hand; a hand still counting an ace as 11 is *soft*.

In [ ]:
from blackjack.game.hand import calculate_score, has_usable_ace, is_bust


def make(*ranks):
    return [{"rank": r, "suit": "Spades"} for r in ranks]


for ranks in [("10", "7"), ("Ace", "6"), ("Ace", "9", "5"), ("Ace", "Ace"), ("King", "Queen", "5")]:
    hand = make(*ranks)
    print(f"{str(ranks):32} score={calculate_score(hand):>2}  soft={has_usable_ace(hand)!s:5}  bust={is_bust(hand)}")

## Outcomes

`evaluate_outcome(player, dealer)` returns an `Outcome` enum.

In [ ]:
from blackjack.game.rules import evaluate_outcome

for player, dealer in [(22, 18), (20, 22), (20, 18), (17, 20), (19, 19)]:
    print(f"player {player} vs dealer {dealer}: {evaluate_outcome(player, dealer).value}")

## The dealer

The dealer is silent; `play_dealer` returns the cards it drew.

In [ ]:
from blackjack.game.dealer import play_dealer

dealer_hand = make("7", "5")
drawn = play_dealer(create_deck(random.Random(1)), dealer_hand)
print("drew:", [c["rank"] for c in drawn], "-> final score", calculate_score(dealer_hand))

## Strategies

Every player is a `Strategy`: `decide(GameState) -> Action`.

In [ ]:
from blackjack.ai.base import GameState
from blackjack.ai.baseline import BaselineStrategy
from blackjack.ai.rule_based import RuleBasedStrategy

baseline, rule_based = BaselineStrategy(), RuleBasedStrategy()
print(f"{'state':<56}{'baseline':>10}{'rule-based':>12}")
for state in [GameState(15, 6, False), GameState(15, 10, False), GameState(12, 5, False),
              GameState(18, 10, True), GameState(18, 6, True), GameState(20, 10, False)]:
    print(f"{state!s:<56}{baseline.decide(state).value:>10}{rule_based.decide(state).value:>12}")

## A full round

`play_round` deals and plays player, AI, and dealer, reporting each step through an `emit` callback. Here both players are computer strategies, and the console renderer prints the events.

In [ ]:
from blackjack.game.engine import play_round
from blackjack.ui.console import ConsoleRenderer, render_results

result = play_round(baseline, rule_based, rng=random.Random(7), emit=ConsoleRenderer())
render_results(result)